Using the RoBERTa algorithm to distiguish between human and AI text

Idea is from https://www.nature.com/articles/s41598-025-27377

Source code mostly from https://github.com/shamylafirdoos/Gpt-vs-Human-Text-Classification

In [1]:
# Get dataset
from local_utilities.dataset import get_dataset_train, get_dataset_test, get_dataset_validation

# Train data
X_train, y_train = get_dataset_train()

# Validation data
X_val, y_val = get_dataset_validation()

[nltk_data] Downloading package punkt_tab to /home/tobias/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/home/tobias/.pyenv/versions/genai/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Get CUDA device
from local_utilities.gpu import get_gpu
device = get_gpu()
print(device)

cuda


In [3]:
def tokenize_texts(texts, tokenizer, max_length=256):
    return tokenizer(
        texts,
        max_length=max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )

In [4]:
def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, total_correct = 0, 0
    for batch in loader:
        input_ids, attention_masks, labels = [x.to(device) for x in batch]
        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(input_ids = input_ids, attention_mask=attention_masks, labels=labels)
            loss = outputs.loss

        total_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        preds = torch.argmax(outputs.logits, dim=-1)
        total_correct += (preds == labels).sum().item()
    return total_loss / len(loader), total_correct / len(loader.dataset)

In [5]:
import numpy as np

def evaluate_model(model, loader, loss_fn, device):
    model.eval()
    preds_all, labels_all, probs_all = [], [], []
    total_loss, total_correct = 0, 0

    with torch.no_grad():
        for batch in loader:
            input_ids, attention_masks, labels = [x.to(device) for x in batch]
            outputs = model(input_ids=input_ids, attention_mask=attention_masks, labels=labels)
            total_loss += outputs.loss.item()
            probs = torch.softmax(outputs.logits, dim=-1)[:,1].cpu().numpy()
            preds = torch.argmax(outputs.logits, dim=-1)
            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            probs_all.extend(probs)
            total_correct += (preds == labels).sum().item()
    return total_loss / len(loader), total_correct / len(loader.dataset), np.array(preds_all), np.array(labels_all), np.array(probs_all)


In [6]:
from sklearn.model_selection import train_test_split
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

train_losses = []
val_losses = []

def run_experiment(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Tokenizer training data
    tokenized_training = tokenize_texts(X_train, tokenizer)
    input_ids_training = tokenized_training["input_ids"]
    attention_training = tokenized_training["attention_mask"]
    labels_training = torch.tensor(y_train)
    train_data = TensorDataset(input_ids_training, attention_training, labels_training)
    
    # Tokenize validation data
    tokenized_validation = tokenize_texts(X_val, tokenizer)
    input_ids_val = tokenized_validation["input_ids"]
    attention_val = tokenized_validation["attention_mask"]
    labels_val = torch.tensor(y_val)
    val_data = TensorDataset(input_ids_val, attention_val, labels_val)
    
    metrics_all = []
    conf_matrices = []
    preds_all, labels_all, probs_all = [], [], []

    train_loader = DataLoader(
        dataset = train_data,
        batch_size=96,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )
    val_loader = DataLoader(
        dataset=val_data,
        batch_size=96,
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
    )

    # Model
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(device)
    model = torch.compile(model, mode="max-autotune")
    optimizer = AdamW(model.parameters(), lr=3e-5, fused=True)
    loss_fn = CrossEntropyLoss()

    # Train
    for epoch in range(3):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn, device)
        val_loss, val_acc, _, _, _ = evaluate_model(model, val_loader, loss_fn, device)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f} | Val Loss={val_loss:.4f}, Val Acc={val_acc:.4f}")

    return model, tokenizer

In [7]:
from local_utilities.model import get_roberta_model
model, tokenizer = run_experiment(get_roberta_model())

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1541.31it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consid

Epoch 1: Train Loss=0.1583, Train Acc=0.9308 | Val Loss=0.1508, Val Acc=0.9374
Epoch 2: Train Loss=0.1056, Train Acc=0.9541 | Val Loss=0.1654, Val Acc=0.9392
Epoch 3: Train Loss=0.0824, Train Acc=0.9639 | Val Loss=0.1692, Val Acc=0.9449


In [8]:
"""
import matplotlib.pyplot as plt

x = [i+1 for i in range(len(train_losses))]

plt.figure(figsize=(8, 3))
plt.plot(x, train_losses, label="Train loss")
plt.plot(x, val_losses, label="Validation loss")
plt.legend()
plt.title("RoBERTa losses")
plt.ylabel("Loss")
plt.xlabel("Epochs")
plt.show()
"""

'\nimport matplotlib.pyplot as plt\n\nx = [i+1 for i in range(len(train_losses))]\n\nplt.figure(figsize=(8, 3))\nplt.plot(x, train_losses, label="Train loss")\nplt.plot(x, val_losses, label="Validation loss")\nplt.legend()\nplt.title("RoBERTa losses")\nplt.ylabel("Loss")\nplt.xlabel("Epochs")\nplt.show()\n'

In [9]:
# Save model
from local_utilities.directory import get_roberta_directory

directory = get_roberta_directory()
model.save_pretrained(directory)
tokenizer.save_pretrained(directory)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.54it/s]


('./data/models/roberta_sentences/tokenizer_config.json',
 './data/models/roberta_sentences/tokenizer.json')